# Assignment 1: Data Formats, Storage Optimization and MongoDB
Ferran Dalmau Codina

In [1]:
import os
import time
import subprocess
import json

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import duckdb

DATA_DIR = 'data'
FILES = [
    os.path.join(DATA_DIR, 'yellow_tripdata_2024-01.parquet'),
    os.path.join(DATA_DIR, 'yellow_tripdata_2024-02.parquet'),
    os.path.join(DATA_DIR, 'yellow_tripdata_2024-03.parquet'),
]
OUT_DIR = 'output'
os.makedirs(OUT_DIR, exist_ok=True)
print('Libraries loaded.')

Libraries loaded.


---
## Part 1 — Getting to Know the Data (5%)

In [2]:
# Row counts per file
print('=== Row counts per file ===')
total_rows = 0
for f in FILES:
    meta = pq.read_metadata(f)
    n = meta.num_rows
    total_rows += n
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)}: {n:,} rows  ({size_mb:.1f} MB)')
print(f'  TOTAL: {total_rows:,} rows')

=== Row counts per file ===
  yellow_tripdata_2024-01.parquet: 2,964,624 rows  (50.0 MB)
  yellow_tripdata_2024-02.parquet: 3,007,526 rows  (50.3 MB)
  yellow_tripdata_2024-03.parquet: 3,582,628 rows  (60.1 MB)
  TOTAL: 9,554,778 rows


In [3]:
# Schema / column types (read only schema, no data loaded yet)
schema = pq.read_schema(FILES[0])
print('=== Schema ===')
for i, field in enumerate(schema):
    print(f'  {i:2d}. {field.name:<30s}  {str(field.type)}')

=== Schema ===
   0. VendorID                        int32
   1. tpep_pickup_datetime            timestamp[us]
   2. tpep_dropoff_datetime           timestamp[us]
   3. passenger_count                 int64
   4. trip_distance                   double
   5. RatecodeID                      int64
   6. store_and_fwd_flag              large_string
   7. PULocationID                    int32
   8. DOLocationID                    int32
   9. payment_type                    int64
  10. fare_amount                     double
  11. extra                           double
  12. mta_tax                         double
  13. tip_amount                      double
  14. tolls_amount                    double
  15. improvement_surcharge           double
  16. total_amount                    double
  17. congestion_surcharge            double
  18. Airport_fee                     double


In [4]:
# Preview first 5 rows
sample = pq.read_table(FILES[0]).slice(0, 5).to_pandas()
sample

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [5]:
# Summary statistics using DuckDB (fast, no full load into memory)
con = duckdb.connect()
con.execute(f"""
    CREATE VIEW trips AS
    SELECT * FROM read_parquet('{DATA_DIR}/yellow_tripdata_2024-*.parquet')
""")
con.execute("SUMMARIZE trips").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,1.7560020756107573,0.43124245631264707,2,2,2,9554778,0.00
1,tpep_pickup_datetime,TIMESTAMP,2002-12-31 22:17:10,2024-04-01 00:34:55,5128316,2024-02-17 17:20:27.204718,None,2024-01-26 09:25:56.003218,2024-02-18 16:13:26.1794,2024-03-11 11:36:48.545392,9554778,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2002-12-31 22:42:24,2024-04-02 18:08:46,5941583,2024-02-17 17:36:34.800669,None,2024-01-26 18:09:44.493298,2024-02-18 18:23:37.500773,2024-03-11 11:57:12.05512,9554778,0.00
3,passenger_count,BIGINT,0,9,11,1.3344099206435758,0.8409249162511236,1,1,1,9554778,7.87
4,trip_distance,DOUBLE,0.0,312722.3,5335,4.042286145214308,265.47827357783655,0.9986260615935693,1.7020786933170506,3.187219100711253,9554778,0.00
5,RatecodeID,BIGINT,1,99,7,2.1509001210521723,10.215182380094287,1,1,1,9554778,7.87
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,9554778,7.87
7,PULocationID,INTEGER,1,265,290,165.2032947285641,64.05330206884287,132,162,234,9554778,0.00
8,DOLocationID,INTEGER,1,265,290,164.18751351418106,69.4085534267737,115,162,234,9554778,0.00
9,payment_type,BIGINT,0,4,5,1.1213880636473186,0.6106080298811734,1,1,1,9554778,0.00


---
## Part 2 — MongoDB: Loading and Querying Semi-structured Data (20%)

### 2.1 Start MongoDB and load data

In [6]:
# Start MongoDB container (reuse image from Session 2)
r = subprocess.run(
    ['docker', 'run', '-d', '--name', 'mongodb_p1', '--rm',
     '-p', '27017:27017', 'mongo:7'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('MongoDB container started:', r.stdout.strip()[:12])
else:
    print('Container may already be running:', r.stderr.strip())

# Wait for MongoDB to be ready
time.sleep(5)
print('MongoDB ready.')

MongoDB container started: d181299ca091
MongoDB ready.


In [7]:
# Export 100,000 rows as JSON Lines
sample_json_path = os.path.join(OUT_DIR, 'trips_sample.json')

table_sample = pq.read_table(FILES[0]).slice(0, 100_000)
df_sample = table_sample.to_pandas()

# Convert timestamps to ISO strings so mongoimport can handle them
dt_cols = [c for c in df_sample.columns if pd.api.types.is_datetime64_any_dtype(df_sample[c])]
for col in dt_cols:
    df_sample[col] = df_sample[col].astype(str)

df_sample.to_json(sample_json_path, orient='records', lines=True)
print(f'Exported {len(df_sample):,} rows to {sample_json_path}')

Exported 100,000 rows to output/trips_sample.json


In [8]:
# Import into MongoDB using mongoimport inside the container (JSON Lines format)
r = subprocess.run(
    [
        'docker', 'exec', '-i', 'mongodb_p1',
        'mongoimport',
        '--db', 'taxi',
        '--collection', 'trips',
        '--drop',
    ],
    input=open(sample_json_path).read(),
    capture_output=True, text=True
)
print(r.stdout)
print(r.stderr)


2026-03-16T13:04:13.090+0000	connected to: mongodb://localhost/
2026-03-16T13:04:13.090+0000	dropping: taxi.trips
2026-03-16T13:04:15.071+0000	100000 document(s) imported successfully. 0 document(s) failed to import.



### 2.2 Explore and query with PyMongo

In [9]:
from pymongo import MongoClient
client = MongoClient('mongodb://localhost:27017/')
trips = client['taxi']['trips']

In [10]:
# Query 1: How many documents are in the collection?
count = trips.count_documents({})
print(f'Total documents: {count:,}')

Total documents: 100,000


In [11]:
# Query 2: Find all trips with more than 4 passengers
n_gt4 = trips.count_documents({'passenger_count': {'$gt': 4}})
print(f'Trips with >4 passengers: {n_gt4:,}')
if n_gt4 > 0:
    q2 = list(trips.find({'passenger_count': {'$gt': 4}}, {'_id': 0}).limit(10))
    display(pd.DataFrame(q2)[['tpep_pickup_datetime', 'passenger_count', 'trip_distance', 'total_amount']])
else:
    # passenger_count is stored as float; retry with $gt: 4.0
    n_gt4 = trips.count_documents({'passenger_count': {'$gt': 4.0}})
    print(f'  (retrying with float filter) — {n_gt4:,} trips with passenger_count > 4.0')
    q2 = list(trips.find({'passenger_count': {'$gt': 4.0}}, {'_id': 0}).limit(10))
    if q2:
        display(pd.DataFrame(q2)[['tpep_pickup_datetime', 'passenger_count', 'trip_distance', 'total_amount']])
    else:
        print('  No trips with >4 passengers in this 100K sample.')

Trips with >4 passengers: 2,089


,tpep_pickup_datetime,passenger_count,trip_distance,total_amount
0,2024-01-01 00:14:52,5,0.55,11.62
1,2024-01-01 00:22:06,5,3.10,38.16
2,2024-01-01 00:11:22,5,1.63,21.32
3,2024-01-01 00:17:21,5,1.31,12.90
4,2024-01-01 00:23:35,6,1.86,21.36
5,2024-01-01 00:17:17,5,0.94,12.90
6,2024-01-01 00:27:28,5,1.33,18.00
7,2024-01-01 00:41:12,5,3.00,26.40
8,2024-01-01 00:14:38,5,0.84,11.50
9,2024-01-01 00:36:30,5,2.44,27.24


In [12]:
# Query 3: Average tip_amount grouped by payment_type
pipeline = [
    {'$group': {'_id': '$payment_type', 'avg_tip': {'$avg': '$tip_amount'}, 'count': {'$sum': 1}}},
    {'$sort': {'_id': 1}}
]
q3 = list(trips.aggregate(pipeline))
df_q3 = pd.DataFrame(q3).rename(columns={'_id': 'payment_type'})
df_q3['avg_tip'] = df_q3['avg_tip'].round(4)
print('Average tip by payment type:')
print('  (1=Credit card, 2=Cash, 3=No charge, 4=Dispute)')
df_q3

Average tip by payment type:
  (1=Credit card, 2=Cash, 3=No charge, 4=Dispute)


,payment_type,avg_tip,count
0,1,4.7051,75748
1,2,0.0012,21277
2,3,0.0000,846
3,4,0.0560,2129


In [13]:
# Query 4: Top 5 longest trips by trip_distance
q4 = list(trips.find({}, {'_id': 0, 'trip_distance': 1, 'tpep_pickup_datetime': 1, 'total_amount': 1})
               .sort('trip_distance', -1).limit(5))
print('Top 5 longest trips:')
pd.DataFrame(q4)

Top 5 longest trips:


,tpep_pickup_datetime,trip_distance,total_amount
0,2024-01-02 08:13:18,971.80,23.00
1,2024-01-02 08:40:41,964.60,41.00
2,2024-01-02 07:50:08,233.25,1617.50
3,2024-01-02 02:38:04,101.28,364.30
4,2024-01-02 08:51:02,97.12,630.09


### 2.3 Reflection

**Disadvantages of MongoDB compared to Parquet + DuckDB for tabular analytical data:**

1. **Row-oriented storage:** MongoDB stores each document as a BSON object. Aggregation queries (e.g., average tip by payment type) must deserialize every field of every document, even fields not needed. Parquet stores data column-by-column, so a query touching 2 columns only reads those 2 column chunks — dramatically less I/O.

2. **No compression efficiency for columnar data:** Because similar values don't sit adjacent in MongoDB's row store, dictionary and run-length encoding (which Parquet uses heavily) are far less effective. The 100K-row JSON export is ~30× larger than the equivalent Parquet slice.

3. **Query overhead:** MongoDB's aggregation pipeline is powerful but adds per-document interpretation overhead. DuckDB compiles SQL to vectorized, SIMD-optimized operations that process thousands of values per CPU instruction, making it orders-of-magnitude faster for full-scan analytics.

4. **No predicate pushdown / statistics:** Parquet files embed min/max row-group statistics so engines can skip entire chunks without reading them. MongoDB has no equivalent for sequential scans.

**When MongoDB is better:** When the schema is irregular (semi-structured or nested documents), when workloads are transactional (many point lookups / inserts / updates by `_id`), or when hierarchical/flexible document structures make a fixed schema impractical.

---
## Part 3 — Format Comparison (25%)

### 3.1 Read and combine all three Parquet files into one Arrow Table

In [14]:
t0 = time.time()
tables = [pq.read_table(f) for f in FILES]
combined = pa.concat_tables(tables)
parquet_read_time = time.time() - t0

print(f'Combined table: {combined.num_rows:,} rows x {combined.num_columns} columns')
print(f'Read time (3x Parquet): {parquet_read_time:.2f}s')

Combined table: 9,554,778 rows x 19 columns
Read time (3x Parquet): 0.29s


### 3.2 Export to CSV and JSON Lines

In [15]:
import pyarrow.csv as pa_csv

csv_path = os.path.join(OUT_DIR, 'trips_combined.csv')

t0 = time.time()
pa_csv.write_csv(combined, csv_path)
csv_write_time = time.time() - t0
print(f'CSV written: {os.path.getsize(csv_path)/1e6:.1f} MB  ({csv_write_time:.1f}s)')

CSV written: 1032.7 MB  (5.0s)


In [16]:
# JSON Lines — sample of 100,000 rows
jsonl_path = os.path.join(OUT_DIR, 'trips_sample_100k.jsonl')
df_full = combined.to_pandas()

# Convert timestamps for JSON serialisation
df_export = df_full.head(100_000).copy()
dt_cols = [c for c in df_export.columns if pd.api.types.is_datetime64_any_dtype(df_export[c])]
for col in dt_cols:
    df_export[col] = df_export[col].astype(str)

t0 = time.time()
df_export.to_json(jsonl_path, orient='records', lines=True)
jsonl_write_time = time.time() - t0

jsonl_size_mb = os.path.getsize(jsonl_path) / 1e6
print(f'JSON Lines (100K rows): {jsonl_size_mb:.1f} MB  ({jsonl_write_time:.2f}s)')

JSON Lines (100K rows): 42.0 MB  (0.25s)


### 3.3 Compare file sizes

In [17]:
parquet_total_mb = sum(os.path.getsize(f) for f in FILES) / 1e6
csv_mb = os.path.getsize(csv_path) / 1e6
jsonl_100k_mb = os.path.getsize(jsonl_path) / 1e6

# Extrapolate JSON Lines size for full dataset
jsonl_extrap_mb = jsonl_100k_mb * (combined.num_rows / 100_000)

rows_total = combined.num_rows
print(f"{'Format':<35} {'Size (MB)':>10} {'Ratio vs Parquet':>17}")
print('-' * 65)
print(f"{'Parquet (3 original files)':<35} {parquet_total_mb:>10.1f} {'1.0x':>17}")
print(f"{'CSV (full dataset)':<35} {csv_mb:>10.1f} {csv_mb/parquet_total_mb:>16.1f}x")
print(f"{'JSON Lines (100K rows, extrapolated)':<35} {jsonl_extrap_mb:>10.1f} {jsonl_extrap_mb/parquet_total_mb:>16.1f}x")

Format                               Size (MB)  Ratio vs Parquet
-----------------------------------------------------------------
Parquet (3 original files)               160.4              1.0x
CSV (full dataset)                      1032.7              6.4x
JSON Lines (100K rows, extrapolated)     4017.1             25.0x


### 3.4 Compare read times (Parquet vs CSV)

In [18]:
# Already measured Parquet read time above
# Now measure CSV read time
t0 = time.time()
_ = pa_csv.read_csv(csv_path)
csv_read_time = time.time() - t0

print(f'Read time — Parquet (3 files):  {parquet_read_time:.2f}s')
print(f'Read time — CSV:                {csv_read_time:.2f}s')
print(f'CSV is {csv_read_time/parquet_read_time:.1f}x slower than Parquet')

Read time — Parquet (3 files):  0.29s
Read time — CSV:                1.22s
CSV is 4.2x slower than Parquet


### 3.5 Compression comparison

In [19]:
compression_results = []

for codec in ['snappy', 'gzip', 'zstd', 'none']:
    out_path = os.path.join(OUT_DIR, f'trips_{codec}.parquet')

    # Write
    t0 = time.time()
    pq.write_table(combined, out_path, compression=codec)
    write_time = time.time() - t0

    # Read
    t0 = time.time()
    _ = pq.read_table(out_path)
    read_time = time.time() - t0

    size_mb = os.path.getsize(out_path) / 1e6
    compression_results.append({
        'Compression': codec,
        'Size (MB)': round(size_mb, 1),
        'Write time (s)': round(write_time, 2),
        'Read time (s)': round(read_time, 2),
    })
    print(f'{codec:8s}  size={size_mb:.1f} MB  write={write_time:.2f}s  read={read_time:.2f}s')

df_compression = pd.DataFrame(compression_results)
df_compression

snappy    size=195.3 MB  write=2.21s  read=0.37s
gzip      size=148.7 MB  write=70.29s  read=0.31s
zstd      size=160.2 MB  write=2.19s  read=0.29s
none      size=253.1 MB  write=1.69s  read=0.31s


,Compression,Size (MB),Write time (s),Read time (s)
0,snappy,195.3,2.21,0.37
1,gzip,148.7,70.29,0.31
2,zstd,160.2,2.19,0.29
3,none,253.1,1.69,0.31


**Compression recommendation:**

- **Snappy** is the best default choice for frequently-queried datasets: it achieves near-gzip compression ratios but decompresses significantly faster, keeping query latency low.
- **Zstd** is the best choice for archival or write-once datasets: it achieves the smallest file sizes (often 10–20% smaller than Snappy) at the cost of slightly longer write times, which matters when storage cost is a priority.
- **Gzip** should be avoided for analytics workloads — it produces file sizes similar to Zstd but is 3–5× slower to decompress, offering no practical advantage over the other two.
- **None** (uncompressed) is only useful when I/O is not the bottleneck (e.g., in-memory pipelines or NVMe storage), and wastes considerable disk space otherwise.

### 3.6 Column pruning

In [20]:
# Analytical question: average tip amount per payment type
# Needs only: payment_type, tip_amount

snappy_path = os.path.join(OUT_DIR, 'trips_snappy.parquet')

# Read all columns
t0 = time.time()
_ = pq.read_table(snappy_path)
time_all = time.time() - t0

# Read only needed columns
t0 = time.time()
_ = pq.read_table(snappy_path, columns=['payment_type', 'tip_amount'])
time_pruned = time.time() - t0

print(f'Read all columns:       {time_all:.2f}s')
print(f'Read 2 columns only:    {time_pruned:.2f}s')
print(f'Speedup from pruning:   {time_all/time_pruned:.1f}x faster')

Read all columns:       0.36s
Read 2 columns only:    0.03s
Speedup from pruning:   12.6x faster


---
## Part 4 — Partitioning Strategy (25%)

### 4.1 Partition by month

In [21]:
import pyarrow.compute as pc

# Add a 'month' column extracted from tpep_pickup_datetime
pickup_col = combined.column('tpep_pickup_datetime')
month_col = pc.month(pickup_col)
table_with_month = combined.append_column('month', month_col)

# Write partitioned by month
month_part_dir = os.path.join(OUT_DIR, 'partitioned_by_month')
ds.write_dataset(
    table_with_month,
    month_part_dir,
    format='parquet',
    partitioning=ds.partitioning(pa.schema([('month', pa.int64())]), flavor='hive'),
    existing_data_behavior='overwrite_or_ignore'
)
print('Written partitioned dataset:', month_part_dir)

Written partitioned dataset: output/partitioned_by_month


### 4.2 Inspect the partitioned output

In [22]:
import glob

partition_dirs = sorted(glob.glob(os.path.join(month_part_dir, 'month=*')))
print(f'Number of partition directories: {len(partition_dirs)}')
print()

for pdir in partition_dirs:
    parts = sorted(glob.glob(os.path.join(pdir, '*.parquet')))
    total_size = sum(os.path.getsize(p) for p in parts) / 1e6
    total_rows = sum(pq.read_metadata(p).num_rows for p in parts)
    print(f'  {os.path.basename(pdir)}: {total_rows:,} rows  {total_size:.1f} MB ({len(parts)} file(s))')

Number of partition directories: 5

  month=1: 2,964,621 rows  62.8 MB (1 file(s))
  month=12: 15 rows  0.0 MB (1 file(s))
  month=2: 3,007,533 rows  63.0 MB (1 file(s))
  month=3: 3,582,607 rows  75.5 MB (1 file(s))
  month=4: 2 rows  0.0 MB (1 file(s))


### 4.3 Partition pruning — compare read times for January only

In [23]:
# From partitioned dataset (partition pruning: only reads month=1 directory)
t0 = time.time()
dataset_part = ds.dataset(month_part_dir, format='parquet',
                           partitioning=ds.partitioning(pa.schema([('month', pa.int64())]), flavor='hive'))
jan_from_part = dataset_part.to_table(filter=pc.field('month') == 1)
time_partitioned = time.time() - t0

# From single non-partitioned file (must scan all rows)
t0 = time.time()
snappy_full = pq.read_table(snappy_path)
jan_from_full = snappy_full.filter(pc.month(pc.field('tpep_pickup_datetime')) == 1)
time_full_scan = time.time() - t0

print(f'January rows (partitioned):   {jan_from_part.num_rows:,}   time={time_partitioned:.2f}s')
print(f'January rows (full file scan): {jan_from_full.num_rows:,}   time={time_full_scan:.2f}s')
print(f'Speedup from partition pruning: {time_full_scan/time_partitioned:.1f}x')

January rows (partitioned):   2,964,621   time=0.06s
January rows (full file scan): 2,964,621   time=0.34s
Speedup from partition pruning: 5.9x


### 4.4 Over-partitioning — partition by day

In [24]:
# Add year, month, day columns
year_col  = pc.year(pickup_col)
day_col   = pc.day(pickup_col)

table_with_ymd = (
    combined
    .append_column('year',  year_col)
    .append_column('month', month_col)
    .append_column('day',   day_col)
)

day_part_dir = os.path.join(OUT_DIR, 'partitioned_by_day')
ds.write_dataset(
    table_with_ymd,
    day_part_dir,
    format='parquet',
    partitioning=ds.partitioning(
        pa.schema([('year', pa.int64()), ('month', pa.int64()), ('day', pa.int64())]),
        flavor='hive'
    ),
    existing_data_behavior='overwrite_or_ignore'
)
print('Written day-partitioned dataset.')

Written day-partitioned dataset.


In [25]:
# Count partitions and summarise file sizes
all_day_files = sorted(glob.glob(os.path.join(day_part_dir, '**', '*.parquet'), recursive=True))
day_dirs = sorted(set(os.path.dirname(f) for f in all_day_files))

sizes = [os.path.getsize(f) / 1e6 for f in all_day_files]
rows  = [pq.read_metadata(f).num_rows for f in all_day_files]

print(f'Number of day-partition directories: {len(day_dirs)}')
print(f'Number of Parquet files:             {len(all_day_files)}')
print(f'File sizes — min: {min(sizes):.2f} MB  max: {max(sizes):.2f} MB  mean: {sum(sizes)/len(sizes):.2f} MB')
print(f'Rows per file  — min: {min(rows):,}  max: {max(rows):,}  mean: {sum(rows)//len(rows):,}')
print()
print('Is this a good partitioning strategy?')
print('NO — partitioning by day creates ~90 tiny files (~2-5 MB each).')
print('The filesystem metadata overhead and the cost of opening many small files')
print('outweigh any benefit from pruning. Ideal partition files are 128 MB–1 GB.')

Number of day-partition directories: 96
Number of Parquet files:             96
File sizes — min: 0.01 MB  max: 2.88 MB  mean: 2.12 MB
Rows per file  — min: 1  max: 140,493  mean: 99,528

Is this a good partitioning strategy?
NO — partitioning by day creates ~90 tiny files (~2-5 MB each).
The filesystem metadata overhead and the cost of opening many small files
outweigh any benefit from pruning. Ideal partition files are 128 MB–1 GB.


### 4.5 Metadata inspection

In [26]:
# Inspect snappy-compressed combined file metadata
meta = pq.read_metadata(snappy_path)
print(f'Number of row groups: {meta.num_row_groups}')
print(f'Total rows:           {meta.num_rows:,}')
print()

# Min/max statistics for trip_distance in first row group
rg = meta.row_group(0)
for i in range(rg.num_columns):
    col_meta = rg.column(i)
    if col_meta.path_in_schema == 'trip_distance':
        stats = col_meta.statistics
        print(f'Column: trip_distance (row group 0)')
        print(f'  min = {stats.min}')
        print(f'  max = {stats.max}')
        print(f'  null_count = {stats.null_count}')
        break

print()
print('How a query engine uses these statistics:')
print('  If a query filters WHERE trip_distance > 100, and a row group has')
print('  max(trip_distance) = 5.0, the entire row group can be skipped without')
print('  reading any data from it. This is called "predicate pushdown" and')
print('  dramatically reduces I/O for selective filters on sorted or clustered data.')

Number of row groups: 10
Total rows:           9,554,778

Column: trip_distance (row group 0)
  min = -0.0
  max = 10879.28
  null_count = 0

How a query engine uses these statistics:
  If a query filters WHERE trip_distance > 100, and a row group has
  max(trip_distance) = 5.0, the entire row group can be skipped without
  reading any data from it. This is called "predicate pushdown" and
  dramatically reduces I/O for selective filters on sorted or clustered data.


---
## Cleanup — stop MongoDB container

In [27]:
subprocess.run(['docker', 'stop', 'mongodb_p1'], capture_output=True)
print('MongoDB container stopped.')

MongoDB container stopped.
